## Ejercicio 1: Evaluación de solicitud de crédito bancario

Agente que decide si aprueba, aprueba con condiciones o rechaza una solicitud de crédito, en base a la relación entre el monto solicitado y el ingreso mensual, y el historial de mora del cliente.

### Ficha PEAS

| Elemento | Descripción |
|---|---|
| Percepción (S) | ingreso_mensual, monto_solicitado, tiene_historial_moroso |
| Acciones (A) | aprobar, aprobar con condiciones, rechazar |
| Entorno (E) | Sistema de evaluación crediticia de un banco, recibiendo solicitudes con distintos perfiles de riesgo |
| Objetivo | Aprobar créditos viables y limitar el riesgo de impago |
| Medida de desempeño (P) | Proporción de créditos bien clasificados, mora futura evitada, solicitudes atendidas correctamente |

### Justificación de las reglas

La relación monto/ingreso es el indicador más directo de la capacidad de pago: mientras más alta, más compromete el ingreso del solicitante. Por eso se usan dos cortes (0.3 y 0.5) en vez de uno solo, para dejar un rango intermedio donde el crédito no se niega de plano sino que se aprueba con condiciones (por ejemplo, tasa más alta o un aval). El historial moroso se trata como un agravante y no como un filtro absoluto: sube la exigencia de la relación permitida, pero no bloquea automáticamente a alguien con una solicitud pequeña frente a su ingreso, porque eso sería descartar clientes que sí podrían pagar.

In [ ]:
def agente_credito(ingreso_mensual, monto_solicitado, tiene_historial_moroso):
    relacion = monto_solicitado / ingreso_mensual

    if tiene_historial_moroso:
        if relacion <= 0.3:
            return "aprobar con condiciones", f"relacion {relacion:.2f} baja pese al historial moroso, se exige garantia adicional"
        else:
            return "rechazar", f"relacion {relacion:.2f} mayor a 0.3 con historial moroso"
    else:
        if relacion <= 0.3:
            return "aprobar", f"relacion {relacion:.2f} baja y sin historial moroso"
        elif relacion <= 0.5:
            return "aprobar con condiciones", f"relacion {relacion:.2f} moderada, se otorga con condiciones"
        else:
            return "rechazar", f"relacion {relacion:.2f} mayor a 0.5, supera la capacidad de pago"


### Simulación y pruebas

In [ ]:
casos = [
    (3000, 600, False),   # relacion 0.20 -> aprobar
    (3000, 900, False),   # relacion 0.30 (borde) -> aprobar
    (3000, 1200, False),  # relacion 0.40 -> aprobar con condiciones
    (3000, 1500, False),  # relacion 0.50 (borde) -> aprobar con condiciones
    (3000, 1800, False),  # relacion 0.60 -> rechazar
    (3000, 800, True),    # relacion 0.27 con moroso -> aprobar con condiciones
    (3000, 1000, True),   # relacion 0.33 con moroso -> rechazar
]

for ingreso, monto, moroso in casos:
    accion, motivo = agente_credito(ingreso, monto, moroso)
    print(f"ingreso={ingreso}, monto={monto}, moroso={moroso} -> {accion} ({motivo})")


### Visualización

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

np.random.seed(42)
n = 200
ingresos = np.random.uniform(1000, 8000, n)
montos = np.random.uniform(200, 6000, n)
morosos = np.random.choice([True, False], n)

relaciones = []
acciones = []
for ingreso, monto, moroso in zip(ingresos, montos, morosos):
    accion, _ = agente_credito(ingreso, monto, moroso)
    relaciones.append(monto / ingreso)
    acciones.append(accion)

colores = {"aprobar": "green", "aprobar con condiciones": "orange", "rechazar": "red"}
plt.figure(figsize=(8, 5))
for accion in colores:
    idx = [i for i, a in enumerate(acciones) if a == accion]
    plt.scatter(np.array(relaciones)[idx], ingresos[idx], c=colores[accion], label=accion, alpha=0.7)

plt.xlabel("relacion (monto_solicitado / ingreso_mensual)")
plt.ylabel("ingreso_mensual")
plt.title("Decisiones del agente de credito sobre 200 solicitudes simuladas")
plt.legend()
plt.show()
